# Road PCI Intelligence — Step 3C: Multi-Seed Reproducibility & Stability Ablation

This notebook evaluates seed robustness across **Masked Multi-Task Loss** vs **Naive Multi-Task Loss**:
- Runs sequentially:
  1. `masked` loss with Seed 43 (30 epochs)
  2. `masked` loss with Seed 44 (30 epochs)
  3. `naive` loss with Seed 43 (30 epochs)
  4. `naive` loss with Seed 44 (30 epochs)
- **Zero Weights Policy**: Only metrics and training histories are saved (`save_weights=False`).
- Integrates verified Seed 42 baselines (Run A and Run B).
- Reports:
  - Best-epoch selection score: $\frac{\text{Dice}_{\text{crack500}} + \text{Dice}_{\text{pothole}}}{2.0}$
  - Epoch-30 Dice for `crack500`, `pothole`, and `road_crack` at threshold 0.5
  - Road-crack mean Dice over non-empty images (threshold 0.5)
  - Number of empty road-crack val images getting zero predicted pixels (out of 39)
  - Mean $\pm$ Standard Deviation across the 3 seeds for both loss formulations.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, DEPENDENCIES & REPO REPLICATION
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print("Pulling latest GitHub changes...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
import albumentations as A
import segmentation_models_pytorch as smp

from src.train import seed_everything, log_library_versions

seed_everything(42)
log_library_versions()

assert torch.cuda.is_available(), "CRITICAL: CUDA is required for multi-seed training!"
device = torch.device('cuda')
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# 2. RESOLVE MANIFESTS & DATASET ROOTS
# ==============================================================================
import pandas as pd
from src.build_manifest import DATASET_SPECS, resolve_dataset_root

manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
manifest_df = pd.read_csv(manifest_path)
print(f"Loaded segmentation manifest: {len(manifest_df):,} records")

dataset_roots = {}
for src in ['pothole', 'road_crack', 'crack500']:
    spec = DATASET_SPECS[src]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    assert root is not None and root.exists(), f"Could not find dataset root for {src}"
    dataset_roots[src] = root
    print(f"  • {src:10s} root: {root}")


In [ ]:
# ==============================================================================
# 3. SEQUENTIAL MULTI-SEED TRAINING RUNS (SEEDS 43, 44)
# ==============================================================================
import time
from src.train import run_full_training

runs_to_execute = [
    ('masked', 43),
    ('masked', 44),
    ('naive', 43),
    ('naive', 44),
]

all_histories = {}
all_sweeps = {}

t_total_start = time.time()

for mode, s in runs_to_execute:
    run_tag = f"{mode}_seed{s}"
    out_dir_run = Path(f"/kaggle/working/{run_tag}")
    out_dir_run.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "#" * 80)
    print(f"STARTING EXECUTION: Mode = '{mode.upper()}' | Seed = {s}")
    print("#" * 80)
    
    t0_run = time.time()
    model, history, sweep = run_full_training(
        manifest_df=manifest_df,
        dataset_roots=dataset_roots,
        epochs=30,
        loss_mode=mode,
        loss_terms='bce+dice',
        seed=s,
        output_dir=out_dir_run,
        lr=3e-4,
        weight_decay=1e-4,
        batch_size=8,
        samples_per_epoch=600,
        device=device,
        save_weights=False
    )
    t_run = time.time() - t0_run
    print(f"[COMPLETED] {run_tag} in {t_run:.2f}s ({t_run/60:.2f} min)")
    
    all_histories[run_tag] = history
    all_sweeps[run_tag] = sweep
    
    # Free model and GPU cache between runs
    del model
    torch.cuda.empty_cache()

print(f"\nAll 4 seed runs completed in {time.time() - t_total_start:.2f}s")


In [ ]:
# ==============================================================================
# 4. COMPILE MULTI-SEED BENCHMARKS & REPORT COMPREHENSIVE TABLES
# ==============================================================================
import json
import numpy as np

# Seed 42 verified numbers from Run A and Run B
# Masked Seed 42 (Run A): Best Ep 25, Final Ep 30
seed_42_masked_best = {
    'epoch': 25,
    'selection_score': 0.700327,
    'c500_dice': 0.663860,
    'pot_dice': 0.736794,
    'rc_dice': 0.382476,
    'rc_non_empty_dice': 0.525889,
    'rc_empty_clean': 3
}
seed_42_masked_final = {
    'epoch': 30,
    'selection_score': 0.699442,
    'c500_dice': 0.664115,
    'pot_dice': 0.734770,
    'rc_dice': 0.381309,
    'rc_non_empty_dice': 0.523041,
    'rc_empty_clean': 3
}

# Naive Seed 42 (Run B): Best Ep 30, Final Ep 30
seed_42_naive_best = {
    'epoch': 30,
    'selection_score': 0.714003,
    'c500_dice': 0.673412,
    'pot_dice': 0.754593,
    'rc_dice': 0.437878,
    'rc_non_empty_dice': 0.527405,
    'rc_empty_clean': 11
}
seed_42_naive_final = seed_42_naive_best.copy()

def extract_run_stats(history, sweep):
    best_rec = max(history, key=lambda x: x['selection_score'])
    final_rec = history[-1]
    
    # Sweep at threshold 0.5
    sw_05 = next(r for r in sweep if abs(r['threshold'] - 0.5) < 1e-5)
    
    # Best epoch stats
    b_val = best_rec['val_metrics']
    best_stats = {
        'epoch': best_rec['epoch'],
        'selection_score': best_rec['selection_score'],
        'c500_dice': b_val.get('crack500_dataset_dice', 0.0),
        'pot_dice': b_val.get('pothole_dataset_dice', 0.0),
        'rc_dice': b_val.get('road_crack_dataset_dice', 0.0),
        'rc_non_empty_dice': b_val.get('road_crack_non_empty_mean_image_dice', 0.0),
        'rc_empty_clean': b_val.get('road_crack_empty_clean_images', 0)
    }
    
    # Final epoch (Ep 30) stats
    f_val = final_rec['val_metrics']
    final_stats = {
        'epoch': final_rec['epoch'],
        'selection_score': final_rec['selection_score'],
        'c500_dice': f_val.get('crack500_dataset_dice', 0.0),
        'pot_dice': f_val.get('pothole_dataset_dice', 0.0),
        'rc_dice': f_val.get('road_crack_dataset_dice', 0.0),
        'rc_non_empty_dice': f_val.get('road_crack_non_empty_mean_image_dice', 0.0),
        'rc_empty_clean': f_val.get('road_crack_empty_clean_images', 0)
    }
    
    return best_stats, final_stats

# Collect all data
records_best = {
    ('masked', 42): seed_42_masked_best,
    ('naive', 42): seed_42_naive_best
}
records_final = {
    ('masked', 42): seed_42_masked_final,
    ('naive', 42): seed_42_naive_final
}

for mode in ['masked', 'naive']:
    for s in [43, 44]:
        tag = f"{mode}_seed{s}"
        b_s, f_s = extract_run_stats(all_histories[tag], all_sweeps[tag])
        records_best[(mode, s)] = b_s
        records_final[(mode, s)] = f_s

# Save raw records to file
results_payload = {
    'best_epoch': {f"{m}_{s}": v for (m, s), v in records_best.items()},
    'final_epoch': {f"{m}_{s}": v for (m, s), v in records_final.items()}
}
with open('/kaggle/working/multi_seed_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_payload, f, indent=2)

print("=" * 115)
print("MULTI-SEED EVALUATION TABLE — BEST EPOCH SELECTION (Threshold 0.5)")
print("=" * 115)
print(f"{'Mode':<8} | {'Seed':<5} | {'Best Ep':<7} | {'Select Score':<12} | {'CRACK500 Dice':<14} | {'Pothole Dice':<13} | {'RoadCrack Dice':<15} | {'RC NonEmpty':<12} | {'RC Empty 0-Pred':<16}")
print("-" * 115)

for mode in ['masked', 'naive']:
    for s in [42, 43, 44]:
        d = records_best[(mode, s)]
        print(f"{mode:<8} | {s:<5} | {d['epoch']:<7} | {d['selection_score']:<12.4f} | {d['c500_dice']:<14.4f} | {d['pot_dice']:<13.4f} | {d['rc_dice']:<15.4f} | {d['rc_non_empty_dice']:<12.4f} | {d['rc_empty_clean']:2d}/39 ({d['rc_empty_clean']/39*100:4.1f}%)")
    print("-" * 115)

print("\n" + "=" * 115)
print("MULTI-SEED EVALUATION TABLE — FINAL EPOCH 30 (Threshold 0.5)")
print("=" * 115)
print(f"{'Mode':<8} | {'Seed':<5} | {'Epoch':<7} | {'Select Score':<12} | {'CRACK500 Dice':<14} | {'Pothole Dice':<13} | {'RoadCrack Dice':<15} | {'RC NonEmpty':<12} | {'RC Empty 0-Pred':<16}")
print("-" * 115)

for mode in ['masked', 'naive']:
    for s in [42, 43, 44]:
        d = records_final[(mode, s)]
        print(f"{mode:<8} | {s:<5} | {d['epoch']:<7} | {d['selection_score']:<12.4f} | {d['c500_dice']:<14.4f} | {d['pot_dice']:<13.4f} | {d['rc_dice']:<15.4f} | {d['rc_non_empty_dice']:<12.4f} | {d['rc_empty_clean']:2d}/39 ({d['rc_empty_clean']/39*100:4.1f}%)")
    print("-" * 115)

# SUMMARY: MEAN +/- STD
print("\n" + "=" * 115)
print("SUMMARY STATISTICS ACROSS 3 SEEDS (42, 43, 44) — MEAN +/- STD")
print("=" * 115)
print(f"{'Timing / Metric':<24} | {'Masked (Mean +/- Std)':<30} | {'Naive (Mean +/- Std)':<30}")
print("-" * 115)

metrics_keys = [
    ('Selection Score', 'selection_score'),
    ('CRACK500 Dice', 'c500_dice'),
    ('Pothole Dice', 'pot_dice'),
    ('RoadCrack Dice', 'rc_dice'),
    ('RoadCrack Non-Empty Dice', 'rc_non_empty_dice'),
    ('RC Empty Clean Count', 'rc_empty_clean')
]

print("--- BEST EPOCH ---")
for lbl, k in metrics_keys:
    m_vals = [records_best[('masked', s)][k] for s in [42, 43, 44]]
    n_vals = [records_best[('naive', s)][k] for s in [42, 43, 44]]
    m_str = f"{np.mean(m_vals):.4f} +/- {np.std(m_vals):.4f}"
    n_str = f"{np.mean(n_vals):.4f} +/- {np.std(n_vals):.4f}"
    print(f"{lbl:<24} | {m_str:<30} | {n_str:<30}")

print("-" * 115)
print("--- FINAL EPOCH 30 ---")
for lbl, k in metrics_keys:
    m_vals = [records_final[('masked', s)][k] for s in [42, 43, 44]]
    n_vals = [records_final[('naive', s)][k] for s in [42, 43, 44]]
    m_str = f"{np.mean(m_vals):.4f} +/- {np.std(m_vals):.4f}"
    n_str = f"{np.mean(n_vals):.4f} +/- {np.std(n_vals):.4f}"
    print(f"{lbl:<24} | {m_str:<30} | {n_str:<30}")
print("=" * 115)
